# GISLR — 1st-place solution recreation (1D-CNN + Transformer)

**Pipeline stage:** stage 1 (training), parallel to `gislr.1.models.training.ipynb`.
This is a *port*, not a new experiment: it reproduces the Kaggle GISLR 1st-place
solution (hoyso48, public LB ~0.89) end to end — feature pipeline, augmentation,
architecture and training regime — on this repo's canonical split and registry.
Implements **TODO §4.2**.

The reference notebook was recovered from git history
(`git show fd1c7aa:src/gislr.0.competition.entry.1st.ipynb`, removed in `f7be9a1`)
and read in full rather than re-derived from memory, as §4.2 requires.

## Why a separate notebook from `gislr.1.models.training.ipynb`

TODO §4.2 says to add this as a *section* of the shared training notebook. That
was written on the assumption that only the **architecture** differs. Having now
read the reference, it does not: the architecture is one of five changes, and
three of them are incompatible with the shared notebook's data path.

| | `gislr.1.models.training.ipynb` | here |
|---|---|---|
| landmarks | ME_126 / ME_132 / FP_118 | **FP_118** (`POINT_LANDMARKS` verbatim) |
| coordinates | xy, un-normalized | xy, **reference-point normalized** |
| features | 2 ch/landmark (position) | **6 ch/landmark** (position + lag-1 + lag-2) |
| NaN | zeroed at cache-build time | **preserved** (normalization + masking need it) |
| sequence | uniform subsample to 128 | native rate, random crop to 384 |
| augmentation | none | resample · mirror · affine · temporal & spatial mask |
| model | GRU / LSTM / BiLSTM / CausalConv1D | Conv1DBlock ×3 + Transformer, ×2 stages |
| regime | `v2-plateau-300` (ReduceLROnPlateau + early stop) | `fp-onecycle-300` (cosine, no early stop, AWP) |

The shared notebook exists to enforce *all-else-identical* across architectures.
Injecting a run that changes the features, the sequence handling, the
augmentation and the optimizer would destroy exactly the property it protects.
So: **separate notebook, separate config — same canonical split, same registry,
same `meta.json` schema**, so the leaderboard still compares like with like.

## What is shared, and therefore still comparable

- `modules/model/data.py::get_canonical_split` — stratified 90/10, seed 42, 9,448 val.
- `modules/model/registry.py` — one epoch-seconds run folder, `meta.json` schema v3.
- `modules/scripts/eval_gru.py` — canonical per-class eval (extended in this
  change to score 1st-place-preprocessed runs correctly).

## Artifacts

| path | content |
|---|---|
| `data/cache/gislr/features/{train,val}_fp118-xy_nan_{data,offsets}.npy` | NaN-preserving feature cache (~3 GB train) |
| `data/models/<run_id>/meta.json` | run record (schema v3) |
| `data/models/<run_id>/{best,last}.pt` | checkpoints (gitignored); `last.pt` every epoch = auto-resume |
| `data/models/<run_id>/assets/history.json` | per-epoch curves |
| `data/models/<run_id>/assets/landmarks.npy` | the subset's holistic row indices |

## Resumability

`last.pt` is written atomically every epoch with model / optimizer / Lookahead /
scheduler / AWP-step state. An interrupted run resumes in place via its pointer
file; a **finished** run never is (fresh run folder per training start). §5 is
the only long cell, and it is the resumable one — killing the kernel costs at
most one epoch.

## Every cell is independently re-runnable

§1 is the only shared prerequisite (imports, paths, constants). Every other
section re-reads the config from disk and loads its inputs from the cache, so
tweaking a hyperparameter and re-running one cell is enough.

## 0. Read this before running — what this can and cannot deliver

**Three things to be clear-eyed about.**

### 0.1 This model is OFFLINE-ONLY. It is not a deployment candidate.

`CLAUDE.md`: *"Streaming viability drives everything."* This architecture is not
streaming-viable, for reasons that are properties of the reference design, not
of the port:

- the readout is a **global average over the whole sequence**;
- **self-attention is unmasked in time** — frame *t* attends to frame *t+k*;
- the **normalization statistics are whole-sequence** (mean of the reference
  landmark over every frame, std over every frame and landmark);
- the lag differences are **forward** (`dx[t] = x[t+1] - x[t]`).

It is registered with `streaming: false` and belongs beside `BiLSTM`: an
accuracy reference that prices the causality gap. The last two points are
cheaply fixable (`diff_mode: "backward"` is a config knob; the normalization
needs a running estimator, TODO §7.2). The first two need a causal attention
mask and a last-frame readout — a real but bounded follow-up, and the most
valuable thing to do *after* this reproduces.

**The transferable win is the feature pipeline, not the model.** Normalization
and lag features are architecture-independent and are exactly what TODO §7.2/§7.3
plan to add to the streaming models. If this port lands near 0.89 while the GRU
sits at 0.756, that gap is the size of the prize available to §7 — measured
rather than assumed.

### 0.2 A single run on our split should not be expected to hit 0.89

The reported ~0.89 is the **Kaggle leaderboard** score of a **4-seed ensemble**
trained on **all 94,477 videos** (`train_folds(CFG, ['all'])` at seeds 42–45).
This notebook trains **one seed** on our **90/10 canonical split**, holding out
9,448 videos it therefore never sees.

A realistic expectation for a single model here is **~0.84–0.88**, and the
comparison that matters is against the 0.7565 GRU on the *same* split. Reaching
0.89 requires the ensemble (§7), which is 4× the compute and cannot be scored on
the canonical split at all (those models train on the val set).

### 0.3 Cost — measured, ~1.4 h for the full 300 epochs

The reference reports 6–7 h/fold on a TPU v3-8. The committed config was chosen by
measurement on this machine (RTX 4080 Super 16 GB, i7-14700K, 28 logical cores):

| config | ms/step | 300 epochs | peak GPU |
|---|---|---|---|
| batch 512, `num_workers` 0 | 640 | 8.9 h | 7.9 GB |
| **batch 512, `num_workers` 8** | **101** | **1.4 h** | **7.9 GB** |
| batch 512, `num_workers` 16 | 106 | 1.5 h | 7.9 GB |
| batch 256, `num_workers` 12 | 228 | 6.3 h | 4.0 GB |
| batch 1024, `num_workers` 12 | 1858 | 12.9 h | 15.7 GB |
| batch 512, w 8, `awp_delta: 0` | 66 | 0.9 h | 7.9 GB |

Two things drove that, both worth knowing before changing anything:

**1. Length bucketing.** GISLR clip lengths are severely skewed — median **22**
frames, mean 38, p99 **219**, max 537. Padding each batch to its own maximum
saves almost nothing when batches are composed randomly, because the max of a few
hundred draws sits deep in the tail: the first timing run measured a median padded
length of **277** against a median clip of **22**, about 6.4x wasted compute
(worse in wall time, attention being O(T²)).
`features.LengthBucketedBatchSampler` groups similar-length clips, measured
**7.12x → 1.13x** padding overhead. It keeps batch composition random across
epochs — shuffle, sort only within a pool, shuffle the batches — because a
straight sort by length would make the LR schedule and the BatchNorm statistics
see a systematic length ramp.

**2. The dataloader was ~85% of the step.** Once padding was fixed, augmentation
on a single CPU thread dominated: 640 ms/step at `num_workers: 0` versus 101 ms
at 8. Gains flatten past 8 (16 and 20 measure slightly *worse*), so 8 is the
knee — not the maximum the CPU could run.

Dead ends, so you don't repeat them: **`torch.compile` raises `TritonMissing`** —
there is no Triton wheel on Windows, which closes the main GPU-side lever.
**`pin_memory` must stay off** — a padded batch at 512 is ~557 MB, so pinning it
across workers exhausts page-locked host memory (`CUDA error: out of memory` from
the pin thread). **bf16 is not faster** than fp16 here (105 vs 104 ms), though it
is the right fallback if AWP produces NaNs, since it cannot overflow.

**Still run §4b before §5** — it re-measures on the day and warns if padding
overhead exceeds 2x. If the estimate is unacceptable, the levers in
`src/config/gislr.firstplace.json` are, in order: `awp_delta: 0` (0.9 h, costs
accuracy), `epochs`, then `hidden_size`.

### 0.4 Known instability

The reference author reports **NaN loss roughly one run in five** at
`awp_delta: 0.2`. If §5 shows the loss going NaN: set `awp_delta: 0.1`, or
`grad_clip: 1.0`, or lower `lr` — all three are config keys, and the run resumes
from `last.pt`, so a NaN does not cost the whole run.

## 1. Setup

Imports, tunables, resolved paths. The **only** cell other sections depend on.
No hyperparameters live here — they are in
[`src/config/gislr.firstplace.json`](config/gislr.firstplace.json), read at run
time by `modules.model.train_fp.load_fp_config` (same principle as
`gislr.1.models.training.ipynb` §2).

In [ ]:
# =====
# Setup: imports, shared constants, resolved paths
# =====
import json
import time

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from modules.dataset.landmark.subsets import get_subset
from modules.model import data as D
from modules.model import features as F
from modules.model import registry as R
from modules.model.architectures import ARCHS, build_model
from modules.model.train_fp import DEFAULT_CONFIG, load_fp_config, train_firstplace
from modules.paths import CACHE_DIR, MODELS_DIR, gislr_dir

CONFIG_PATH = DEFAULT_CONFIG          # src/config/gislr.firstplace.json
SMOKE_BATCH = 8                       # batch size for the §3 CPU smoke test
TIMING_BATCHES = 30                   # batches to time in §4 before extrapolating

DATA_DIR = gislr_dir()                # GISLR only — never resolve_datasets()
SIGN2IDX = D.load_label_map(DATA_DIR)

print(f"torch {torch.__version__} · CUDA {torch.cuda.is_available()}"
      + (f" · {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else ""))
print(f"gislr data   : {DATA_DIR}")
print(f"feature cache: {F.FEATURES_DIR}")
print(f"registry     : {MODELS_DIR}")
print(f"config       : {CONFIG_PATH}")
print(f"classes      : {len(SIGN2IDX)}")

## 2. Config — the resolved recipe, and how it maps to the reference

Everything the run will use, printed before anything trains. The right-hand
column is the reference's own value so any deviation is visible rather than
buried.

In [ ]:
# =====
# Load + display the training config, mapped against the reference values
# =====
CFG = load_fp_config(CONFIG_PATH)
HYP = CFG["hyp"]

REFERENCE = {  # hoyso48's CFG / get_model defaults, for side-by-side comparison
    "batch_size": "512 (64 x 8 TPU replicas)",
    "lr": "4e-3 (5e-4 x 8 replicas)",
    "hidden_size": 192,
    "num_layers": "2 stages (dim 192); 4 for the '4x' model",
    "dropout": 0.2,
    "kernel_size": 17,
    "num_heads": 4,
    "expand": 2,
    "late_dropout": 0.8,
    "late_dropout_start_epoch": 15,
    "weight_decay": "0.1 (decoupled, lr-scaled)",
    "epochs": "300 (400 in the final solution)",
    "warmup_epochs": 0,
    "lr_min_ratio": "lr_min 1e-6",
    "grad_clip": "None (clipvalue=1 commented out)",
    "label_smoothing": 0.1,
    "awp_delta": 0.2,
    "awp_start_epoch": 15,
    "lookahead_k": 5,
    "lookahead_alpha": 0.5,
}

rows = [{"key": k, "this run": v, "reference": REFERENCE.get(k, "—")}
        for k, v in HYP.items()]
print(f"architecture : {CFG['architecture']}  (streaming={ARCHS[CFG['architecture']].streaming})")
print(f"regime       : {CFG['regime']}")
print(f"subsets      : {CFG['subsets']} · coords {CFG['coords']}")
print(f"features     : {CFG['features']}")
print()
display(pd.DataFrame(rows).set_index("key"))

if HYP["batch_size"] != 512:
    print(f"\nNOTE: batch {HYP['batch_size']} vs the reference's 512, with lr scaled to "
          f"{HYP['lr']:g}. Keep the lr/batch ratio if you change either.")
if CFG["features"]["diff_mode"] != "forward":
    print("\nNOTE: diff_mode is CAUSAL — this is no longer the reference configuration.")

## 3. Verification before training (TODO §4.2 — "which parts are streaming-safe")

Three checks that are cheap on CPU and expensive to discover 40 epochs in:

1. **Feature pipeline** — shapes, channel layout, NaN handling, and that the
   normalization actually removes signer scale.
2. **Mirror closure** — `flip_lr` swaps every left/right landmark pair and the
   subset is closed under it (an unclosed subset would silently corrupt the
   augmentation).
3. **Causality audit** — which parts of the model read the future. This is the
   evidence behind the `streaming: false` registration in §0.1, measured rather
   than asserted.

In [ ]:
# =====
# 3a. Feature pipeline on one real video: shapes, channels, normalization
# =====
CFG = load_fp_config(CONFIG_PATH)
subset = get_subset(CFG["subsets"][0])
train_split, val_split = D.get_canonical_split(DATA_DIR, SIGN2IDX)

raw = F.load_video_raw(DATA_DIR / train_split["path"].iloc[0], subset.array, "xy")
kept = F.drop_empty_frames(raw)
ref_idx = int(np.searchsorted(subset.array, F.REF_LANDMARK))
feats = F.preprocess(kept, ref_idx, CFG["features"]["max_len"],
                     CFG["features"]["diff_mode"])

P = len(subset)
print(f"subset {subset.name}: {P} landmarks · ref landmark {F.REF_LANDMARK} at index {ref_idx}")
print(f"raw      {raw.shape}  NaN {np.isnan(raw).mean():.1%}")
print(f"filtered {kept.shape}  (all-NaN frames dropped)")
print(f"features {feats.shape}  expected (T, {F.CHANNELS_PER_LANDMARK * P}) "
      f"= position {2*P} | lag-1 {2*P} | lag-2 {2*P}")
assert feats.shape[1] == F.CHANNELS_PER_LANDMARK * P
assert np.isfinite(feats).all(), "NaN/inf leaked into the model input"

pos, d1, d2 = feats[:, :2*P], feats[:, 2*P:4*P], feats[:, 4*P:]
print(f"\nchannel stats  position std {pos.std():.3f} · lag-1 std {d1.std():.3f} "
      f"· lag-2 std {d2.std():.3f}")

# the point of the normalization: a scaled/shifted signer must produce the SAME
# features (this is what removes signer size and distance to camera)
scaled = F.preprocess(kept * 0.7 + 0.15, ref_idx, CFG["features"]["max_len"],
                      CFG["features"]["diff_mode"])
delta = np.nanmax(np.abs(scaled - feats))
print(f"scale/shift invariance: max |Δfeature| = {delta:.2e}  "
      f"({'OK' if delta < 1e-3 else 'FAILED — normalization is not doing its job'})")

In [ ]:
# =====
# 3b. Mirror map: subset closed under left/right swap, flip is an involution
# =====
subset = get_subset(load_fp_config(CONFIG_PATH)["subsets"][0])
perm = F.mirror_permutation(subset.array)   # asserts closure internally

n_swapped = int((perm != np.arange(len(perm))).sum())
print(f"{subset.name}: {n_swapped}/{len(perm)} landmarks swap, "
      f"{len(perm) - n_swapped} are midline")
assert np.array_equal(perm[perm], np.arange(len(perm))), "mirror is not an involution"

x = np.random.default_rng(0).random((5, len(subset), 2)).astype(np.float32)
assert np.allclose(F.flip_lr(F.flip_lr(x, perm), perm), x), "flip_lr is not self-inverse"
print("flip_lr: involution OK · x -> 1-x OK · every pair present in the subset")

for name, group in [("left hand", range(468, 489)), ("lips", [61, 291]),
                    ("eyes", [33, 263])]:
    pairs = [(a, F.MIRROR_PAIRS.get(a)) for a in list(group)[:3]]
    print(f"  {name:10s} {pairs}")

In [ ]:
# =====
# 3c. Causality audit — does the model read the future? (evidence for §0.1)
# =====
CFG = load_fp_config(CONFIG_PATH)
subset = get_subset(CFG["subsets"][0])
feature_dim = len(subset) * F.CHANNELS_PER_LANDMARK
hyp = {**CFG["hyp"], "late_dropout_start_step": 0}

model = build_model(CFG["architecture"], feature_dim, len(SIGN2IDX), hyp).eval()
print(f"{ARCHS[CFG['architecture']].model_name}: "
      f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M params, "
      f"feature_dim {feature_dim}")

torch.manual_seed(0)
T = 64
x = torch.randn(2, T, feature_dim)
lengths = torch.tensor([T, T])
with torch.no_grad():
    base = model(x, lengths)
    corrupted = x.clone()
    corrupted[:, T // 2:] = torch.randn_like(corrupted[:, T // 2:])  # rewrite the future
    after = model(corrupted, lengths)
drift = (base - after).abs().max().item()
print(f"\ncorrupting frames {T//2}..{T} changes the logits by {drift:.3e}")
print("=> reads the future (expected: global pooling + unmasked attention). "
      "OFFLINE-ONLY." if drift > 1e-5 else "=> causal.")

# ...but the convolutional half IS causal; verify the depthwise conv in isolation
from modules.model.architectures import CausalDWConv1D
dw = CausalDWConv1D(8, kernel_size=17).eval()
y = torch.randn(1, 40, 8)
with torch.no_grad():
    a = dw(y)[:, :20]
    y2 = y.clone(); y2[:, 20:] = torch.randn_like(y2[:, 20:])
    b = dw(y2)[:, :20]
print(f"CausalDWConv1D: future corruption changes the past by "
      f"{(a-b).abs().max().item():.3e} => causal, so a streaming variant only "
      f"needs a masked attention + last-frame readout")

# masking sanity: padded frames must not change a shorter sequence's logits
with torch.no_grad():
    short = model(x[:, :30], torch.tensor([30, 30]))
    padded = torch.cat([x[:, :30], torch.randn(2, 20, feature_dim)], 1)
    same = model(padded, torch.tensor([30, 30]))
print(f"padding invariance: max |Δlogit| = {(short-same).abs().max().item():.3e} "
      f"(must be ~0, else the mask is leaking)")

## 4. Feature cache + one-epoch timing

The cache is built **once** per (split, subset) and reused by every later run —
skip-if-exists, atomic writes. Roughly **3 GB** for the FP_118 train split.

The timing cell is the one to look at before committing to §5: it runs a handful
of real training steps (AWP included) and extrapolates to the full 300 epochs.
It creates **no** registry run.

In [ ]:
# =====
# 4a. Build the NaN-preserving feature caches (skip-if-exists, ~3 GB train)
# =====
from tqdm.auto import tqdm

CFG = load_fp_config(CONFIG_PATH)
subset = get_subset(CFG["subsets"][0])
train_split, val_split = D.get_canonical_split(DATA_DIR, SIGN2IDX)
print(f"canonical split: {len(train_split)} train / {len(val_split)} val")

for split_df, prefix in ((train_split, "train"), (val_split, "val")):
    bar = tqdm(total=len(split_df), desc=f"cache {prefix}/{subset.name}", dynamic_ncols=True)
    paths = F.build_nan_cache(split_df, prefix, subset, CFG["coords"], DATA_DIR,
                              progress=lambda done, total: bar.update(done - bar.n))
    bar.update(bar.total - bar.n); bar.close()
    print(f"  {prefix}: {[p.name for p in paths]}")

In [ ]:
# =====
# 4b. Time a few real training steps and extrapolate to the full run
# =====
from torch.utils.data import DataLoader
from modules.model.optim import AWP, Lookahead, cosine_one_cycle

CFG, HYP = load_fp_config(CONFIG_PATH), load_fp_config(CONFIG_PATH)["hyp"]
subset = get_subset(CFG["subsets"][0])
train_split, _ = D.get_canonical_split(DATA_DIR, SIGN2IDX)
tr_data, tr_off = F.build_nan_cache(train_split, "train", subset, CFG["coords"], DATA_DIR)

ds = F.FirstPlaceDataset(train_split, tr_data, tr_off, subset,
                         augment_data=CFG["features"]["augment"],
                         max_len=CFG["features"]["max_len"],
                         diff_mode=CFG["features"]["diff_mode"],
                         mmap=HYP["num_workers"] > 0)
sampler = F.LengthBucketedBatchSampler(F.cached_lengths(tr_off), HYP["batch_size"],
                                       shuffle=True, drop_last=True)
# pin_memory stays False: a padded batch at 512 is ~557 MB and pinning it across
# workers exhausts page-locked host memory (CUDA OOM from the pin thread).
loader = DataLoader(ds, batch_sampler=sampler, collate_fn=F.collate_fn,
                    num_workers=HYP["num_workers"],
                    persistent_workers=HYP["num_workers"] > 0,
                    prefetch_factor=2 if HYP["num_workers"] > 0 else None,
                    pin_memory=False)
steps_per_epoch = len(loader)

device = torch.device("cuda")
feature_dim = len(subset) * F.CHANNELS_PER_LANDMARK
model = build_model(CFG["architecture"], feature_dim, len(SIGN2IDX),
                    {**HYP, "late_dropout_start_step": 0}).to(device)
opt = torch.optim.RAdam(model.parameters(), lr=HYP["lr"],
                        weight_decay=HYP["weight_decay"], decoupled_weight_decay=True)
lookahead = Lookahead(opt, k=HYP["lookahead_k"], alpha=HYP["lookahead_alpha"])
awp = AWP(model, delta=HYP["awp_delta"], start_step=0)   # worst case: AWP on from step 0
crit = torch.nn.CrossEntropyLoss(label_smoothing=HYP["label_smoothing"])
scaler = torch.amp.GradScaler("cuda")

seq_lens, t0 = [], None
model.train()
for i, (feats, lengths, labels) in enumerate(loader):
    if i == TIMING_BATCHES:
        break
    if i == 5:                      # skip warmup/cudnn autotune batches
        torch.cuda.synchronize(); t0 = time.perf_counter()
    seq_lens.append(int(lengths.max()))
    feats, labels = feats.to(device), labels.to(device)
    opt.zero_grad(set_to_none=True)
    with torch.amp.autocast("cuda"):
        loss = crit(model(feats, lengths), labels)
    scaler.scale(loss).backward()
    if awp.active(0) and awp.perturb():
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            adv = crit(model(feats, lengths), labels)
        scaler.scale(adv).backward()
        awp.restore()
    scaler.unscale_(opt)
    scaler.step(opt); scaler.update(); lookahead.sync()
torch.cuda.synchronize()

per_step = (time.perf_counter() - t0) / (TIMING_BATCHES - 5)
epoch_min = per_step * steps_per_epoch / 60
raw_len = F.cached_lengths(tr_off)
overhead = np.mean(seq_lens) / np.clip(raw_len, None, CFG["features"]["max_len"]).mean()
print(f"batch {HYP['batch_size']} · {steps_per_epoch} steps/epoch · "
      f"padded length: median {int(np.median(seq_lens))}, max {max(seq_lens)} "
      f"(cap {CFG['features']['max_len']})")
print(f"clip lengths: median {int(np.median(raw_len))}, mean {raw_len.mean():.0f}, "
      f"p99 {int(np.percentile(raw_len, 99))}  ->  padding overhead {overhead:.2f}x")
if overhead > 2:
    print("  WARNING: padded length far above the typical clip — length bucketing "
          "is not taking effect; check LengthBucketedBatchSampler is in use.")
print(f"{per_step*1000:.0f} ms/step (AWP on)  ->  ~{epoch_min:.2f} min/epoch")
print(f"\nESTIMATE for {HYP['epochs']} epochs: {epoch_min*HYP['epochs']/60:.1f} h "
      f"(+ validation)")
print(f"  with awp_delta=0 it would be roughly {epoch_min*HYP['epochs']/60/2:.1f} h")
print(f"  peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.1f} GB of "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")
del model, opt, lookahead, scaler; torch.cuda.empty_cache()

## 5. Train (TODO §4.2)

One registry run per configured subset. **This is the long cell** — check §4b's
estimate first.

Everything is config-driven (`src/config/gislr.firstplace.json`); nothing here is
tunable in the cell, deliberately. Interrupting is safe: `last.pt` is written
every epoch and re-running this cell resumes the same run folder.

In [ ]:
# =====
# 5. Train the 1st-place port (resumable; one run folder per subset)
# =====
RUN_DIRS = train_firstplace()
for name, run_dir in RUN_DIRS.items():
    meta = R.load_meta(run_dir)
    print(f"{name}: run {run_dir.name} · best val {meta['metrics']['train_val_acc']:.4f} "
          f"· {meta['training']['epochs_trained']} epochs "
          f"· {meta['training']['wall_time_min']:.0f} min")

## 6. Results — learning curves and the comparison that matters

The 1st-place port against this repo's existing leaderboard, on the **same
canonical split and metric**. `train_val_acc` is the training-loop number; the
canonical figure arrives in §7.

In [ ]:
# =====
# 6a. Learning curves for this run (train/val loss + accuracy, LR)
# =====
CFG = load_fp_config(CONFIG_PATH)
subset_name = CFG["subsets"][0]
run_dir = R.pointer_run_dir(f"{CFG['dataset']}_{CFG['architecture']}_"
                            f"{D.subset_tag(subset_name, CFG['coords'])}")
history = json.loads((run_dir / "assets" / "history.json").read_text())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
epochs = range(1, len(history["train_loss"]) + 1)
axes[0].plot(epochs, history["train_loss"], label="train")
axes[0].plot(epochs, history["val_loss"], label="val")
axes[0].set_title("loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(epochs, history["train_acc"], label="train")
axes[1].plot(epochs, history["val_acc"], label="val")
axes[1].axhline(0.7565, ls="--", c="grey", label="best GRU (canonical)")
axes[1].set_title("accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
axes[2].plot(epochs, history["lr"]); axes[2].set_yscale("log")
axes[2].set_title("learning rate (cosine one-cycle)"); axes[2].set_xlabel("epoch")
fig.suptitle(f"{run_dir.name} · {CFG['architecture']} · {subset_name}")
fig.tight_layout(); plt.show()

best_i = int(np.argmax(history["val_acc"]))
gap = history["train_acc"][best_i] - history["val_acc"][best_i]
print(f"best epoch {best_i+1}: val {history['val_acc'][best_i]:.4f} · "
      f"train {history['train_acc'][best_i]:.4f} · train/val gap {gap:.3f}")
print("Compare that gap with the 0.155-0.24 of the un-augmented runs "
      "(docs/logs/daily/2026-07-19.md): augmentation + AWP should shrink it.")

In [ ]:
# =====
# 6b. Leaderboard — 1st-place port vs every existing registry run
# =====
import duckdb

lb = duckdb.sql(f"""
    SELECT architecture, subset, coords, streaming,
           round(coalesce(metrics.overall_accuracy, metrics.train_val_acc), 4) AS acc,
           metrics.eval_status AS eval,
           training.regime AS regime, n_params, run_id
    FROM read_json_auto('{R.META_GLOB.replace(chr(92), '/')}')
    WHERE dataset = 'gislr'
    ORDER BY acc DESC
    LIMIT 15
""").df()
display(lb)
print("`acc` is the canonical overall_accuracy where available, else the "
      "training-loop val accuracy — see the `eval` column.")

## 7. Canonical evaluation handoff

The training-loop number is not leaderboard-eligible. `eval_gru.py` re-scores the
run from raw parquet on the canonical val set and promotes `meta.json` to
`eval_status: "canonical"`.

It dispatches on the checkpoint's `features` key, so a run trained here is scored
through the 1st-place preprocessing rather than the default one — feeding it the
default 2-channel un-normalized features would report near-chance accuracy.

Then rebuild the index and run `gislr.2.models.evaluation.ipynb` for the full
leaderboard, confusion matrices and the TFLite export.

In [ ]:
# =====
# 7. Print the canonical-eval commands for every run produced above
# =====
CFG = load_fp_config(CONFIG_PATH)
for name in CFG["subsets"]:
    rd = R.pointer_run_dir(f"{CFG['dataset']}_{CFG['architecture']}_"
                           f"{D.subset_tag(name, CFG['coords'])}")
    if rd is not None:
        print(R.eval_command(rd))
print("\n.venv/Scripts/python.exe src/modules/scripts/build_model_index.py")
print("\nThen: src/gislr.2.models.evaluation.ipynb (leaderboard, confusion "
      "matrices, TFLite export).")
print("\nNOTE: TFLite export via modules/model/keras_export.py has NOT been "
      "extended to this architecture (it rebuilds GRU/LSTM/BiLSTM/CausalConv1D "
      "in native Keras). Exporting the port needs Keras equivalents of "
      "Conv1DBlock/ECA/TransformerBlock — out of scope here, filed in TODO §4.2.")

## 8. Next steps

Ordered by what each buys:

1. **Canonical-eval the run** (§7) and record the gap against the 0.7565 GRU.
   That number is the size of the prize for TODO §7.
2. **Ablate the feature pipeline on the GRU** — normalization alone, then
   normalization + lag features, all else identical, via
   `gislr.1.models.training.ipynb`. This is TODO §7.2/§7.3, and it is what
   converts this offline result into a *deployable* gain. If most of the jump
   comes from features rather than the architecture, the streaming models get
   most of it too.
3. **Causal variant of this architecture** — masked attention + last-frame
   readout + `diff_mode: "backward"` + a running-statistics normalizer. That
   turns an offline reference into a deployment candidate and directly prices
   the causality gap for a modern architecture.
4. **The 4-seed ensemble on all data** (§0.2) only if a Kaggle leaderboard number
   is the actual goal — it cannot be scored on the canonical split.